1. Install and API key

In [1]:
!pip install openai pandas scikit-learn tqdm

In [2]:
# OPENAI_API_KEY

import os

os.environ["OPENAI_API_KEY"] = "your_key"

In [3]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

models = client.models.list()

for m in models.data:
    print(m.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-live-transcribe
gpt-5.6-luna
gpt-realtime-2.1
gpt-realtime-2.1-mini
gpt-transcribe
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-re

In [4]:
import os
import json
import time
import pandas as pd
import numpy as np

from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics import precision_recall_fscore_support

# OPENAI_API_KEY
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

RANDOM_SEED = 42
MODEL_NAME = "gpt-5.4-mini"  # Use a cost-effective model available in your account

2. Load pairwise dataset with the same Bin split

In [5]:
# Load the same bin-split dataset used for ML, CNN, BERT, and SciBERT
df = pd.read_csv("pairwise_gold_dataset_with_bins.csv", encoding="utf-8-sig")

print(df["Label"].value_counts())
print(pd.crosstab(df["Bin"], df["Label"]))

Label
0    2830
1     566
Name: count, dtype: int64
Label    0   1
Bin           
0      283  57
1      283  57
2      283  57
3      283  57
4      283  57
5      283  57
6      283  56
7      283  56
8      283  56
9      283  56


3. Define JSON schema for structured output

In [6]:
# Structured Outputs are useful here because we want every API response to contain the same fields: label and probability_match.
# The reason field is removed to reduce output tokens and speed up API calls.

classification_schema = {
    "type": "object",
    "properties": {
        "label": {
            "type": "integer",
            "enum": [0, 1],
            "description": "1 if the two organization names refer to the same organization, otherwise 0."
        },
        "probability_match": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
            "description": "Estimated probability that the pair is a match."
        }
    },
    "required": ["label", "probability_match"],
    "additionalProperties": False
}


4. Zero-shot prompt

In [7]:
ZERO_SHOT_INSTRUCTIONS = """
You are an expert in organization name disambiguation and entity resolution.

Your task is to decide whether two organization name strings refer to the same real-world organization.

Return:
- label = 1 if they refer to the same organization.
- label = 0 if they refer to different organizations.
- probability_match = a number from 0 to 1.

Consider:
- multilingual names and translations
- abbreviations and acronyms
- punctuation, spelling variants, and word order differences
- parent/sub-unit relationships
- government ministries, universities, foundations, councils, agencies, and research institutes

Be conservative:
- If one name is only a broad parent organization and the other is a specific sub-unit, classify as non-match unless they clearly refer to the same entity.
"""

In [8]:
def build_zero_shot_input(name1, name2):
    return f"""
Organization name A:
{name1}

Organization name B:
{name2}

Do these two names refer to the same organization?
"""

5. Few-shot prompt builder

In [9]:
# In few-shot learning, examples are sampled from the training bins within each fold and then applied to the test bin.
# This approach ensures consistency with the evaluation design used in traditional models such as ML, CNN, and BERT.

def build_few_shot_examples(train_df, n_pos=3, n_neg=3):
    # Sample positive and negative examples from training data only
    pos_examples = train_df[train_df["Label"] == 1].sample(
        n=min(n_pos, sum(train_df["Label"] == 1)),
        random_state=RANDOM_SEED
    )
    
    neg_examples = train_df[train_df["Label"] == 0].sample(
        n=min(n_neg, sum(train_df["Label"] == 0)),
        random_state=RANDOM_SEED
    )
    
    examples = pd.concat([pos_examples, neg_examples]).sample(
        frac=1,
        random_state=RANDOM_SEED
    )
    
    example_texts = []
    
    for _, row in examples.iterrows():
        example_texts.append(
            f"""
Example:
Organization name A: {row["OrgName_1"]}
Organization name B: {row["OrgName_2"]}
Correct label: {row["Label"]}
"""
        )
    
    return "\n".join(example_texts)


def build_few_shot_input(name1, name2, examples_text):
    return f"""
Here are labeled examples:

{examples_text}

Now classify the following pair.

Organization name A:
{name1}

Organization name B:
{name2}

Do these two names refer to the same organization?
"""

6. API call function

In [10]:
def classify_pair_with_chatgpt(name1, name2, mode="zero_shot", examples_text=None, max_retries=3):
    # Build prompt depending on zero-shot or few-shot mode
    if mode == "zero_shot":
        user_input = build_zero_shot_input(name1, name2)
    elif mode == "few_shot":
        user_input = build_few_shot_input(name1, name2, examples_text)
    else:
        raise ValueError("mode must be either 'zero_shot' or 'few_shot'")
    
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                instructions=ZERO_SHOT_INSTRUCTIONS,
                input=user_input,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "organization_match_classification",
                        "schema": classification_schema,
                        "strict": True
                    }
                }
            )
            
            result = json.loads(response.output_text)
            
            return {
                "PredictedLabel": int(result["label"]),
                "Probability_Match": float(result["probability_match"])
            }
        
        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    "PredictedLabel": np.nan,
                    "Probability_Match": np.nan,
                    "API_Error": str(e)
                }
            
            # Exponential backoff before retrying
            time.sleep(2 ** attempt)


7. Run zero-shot or few-shot evaluation

In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed

RUN_MODE = "zero_shot"    # Options: "zero_shot" or "few_shot"
MAX_WORKERS = 3           # Start with 3. Increase to 5 only if you do not hit rate limits.
SAMPLE_PER_BIN = None       # Use 20 for testing. Set to None to run all rows in each test bin.

all_predictions = []


def classify_row(row, test_bin, run_mode="zero_shot", examples_text=None):
    # Classify one pair and return a row-level prediction record
    result = classify_pair_with_chatgpt(
        name1=row["OrgName_1"],
        name2=row["OrgName_2"],
        mode=run_mode,
        examples_text=examples_text
    )
    
    output_row = row.to_dict()
    output_row["Fold"] = test_bin
    output_row["Model"] = f"ChatGPT_{run_mode}"
    output_row["TrueLabel"] = row["Label"]
    output_row["PredictedLabel"] = result.get("PredictedLabel", np.nan)
    output_row["Probability_Match"] = result.get("Probability_Match", np.nan)
    output_row["Probability_NonMatch"] = (
        1 - output_row["Probability_Match"]
        if pd.notna(output_row["Probability_Match"])
        else np.nan
    )
    output_row["API_Error"] = result.get("API_Error", None)
    
    return output_row


for test_bin in range(10):
    dev_bin = (test_bin + 1) % 10
    
    train_bins = [
        b for b in range(10)
        if b not in [test_bin, dev_bin]
    ]
    
    train_df = df[df["Bin"].isin(train_bins)].reset_index(drop=True)
    dev_df = df[df["Bin"] == dev_bin].reset_index(drop=True)
    test_df = df[df["Bin"] == test_bin].reset_index(drop=True)
    
    # Optional small sample for API speed/cost testing
    if SAMPLE_PER_BIN is not None:
        test_df = test_df.sample(
            n=min(SAMPLE_PER_BIN, len(test_df)),
            random_state=RANDOM_SEED
        ).reset_index(drop=True)
    
    print(f"\nFold {test_bin}")
    print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
    print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
    print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())
    
    examples_text = None
    
    if RUN_MODE == "few_shot":
        examples_text = build_few_shot_examples(train_df, n_pos=3, n_neg=3)
    
    fold_outputs = []
    
    # Run API calls in parallel. This is much faster than one-by-one sequential calls.
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(
                classify_row,
                row,
                test_bin,
                RUN_MODE,
                examples_text
            )
            for _, row in test_df.iterrows()
        ]
        
        for future in tqdm(as_completed(futures), total=len(futures)):
            fold_outputs.append(future.result())
    
    # Add this fold's outputs to the full prediction list
    all_predictions.extend(fold_outputs)
    
    # Save intermediate results after each fold to avoid losing progress
    pd.DataFrame(all_predictions).to_csv(
        f"chatgpt_{RUN_MODE}_predictions_with_probability_partial.csv",
        index=False,
        encoding="utf-8-sig"
    )

chatgpt_predictions_df = pd.DataFrame(all_predictions)



Fold 0
Train: (2716, 9) {0: 2264, 1: 452}
Dev: (340, 9) {0: 283, 1: 57}
Test: (340, 9) {0: 283, 1: 57}


100%|██████████| 340/340 [01:42<00:00,  3.32it/s]



Fold 1
Train: (2716, 9) {0: 2264, 1: 452}
Dev: (340, 9) {0: 283, 1: 57}
Test: (340, 9) {0: 283, 1: 57}


100%|██████████| 340/340 [01:52<00:00,  3.02it/s]



Fold 2
Train: (2716, 9) {0: 2264, 1: 452}
Dev: (340, 9) {0: 283, 1: 57}
Test: (340, 9) {0: 283, 1: 57}


100%|██████████| 340/340 [01:43<00:00,  3.28it/s]


... (10 outputs left out)


Fold 8
Train: (2718, 9) {0: 2264, 1: 454}
Dev: (339, 9) {0: 283, 1: 56}
Test: (339, 9) {0: 283, 1: 56}


100%|██████████| 339/339 [01:51<00:00,  3.04it/s]



Fold 9
Train: (2717, 9) {0: 2264, 1: 453}
Dev: (340, 9) {0: 283, 1: 57}
Test: (339, 9) {0: 283, 1: 56}


100%|██████████| 339/339 [01:42<00:00,  3.30it/s]


8. Save predictions_with_probability.csv

In [12]:
chatgpt_predictions_df.to_csv(
    f"chatgpt_{RUN_MODE}_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

9. Evaluate match and non-match separately

In [13]:
valid_pred_df = chatgpt_predictions_df.dropna(
    subset=["PredictedLabel", "Probability_Match"]
).copy()

valid_pred_df["PredictedLabel"] = valid_pred_df["PredictedLabel"].astype(int)

fold_results = []

for fold, fold_df in valid_pred_df.groupby("Fold"):
    y_true = fold_df["TrueLabel"].values
    y_pred = fold_df["PredictedLabel"].values
    
    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )
    
    fold_results.append({
        "Fold": fold,
        "Model": f"ChatGPT_{RUN_MODE}",
        
        "Match_Precision": class_precision[1],
        "Match_Recall": class_recall[1],
        "Match_F1": class_f1[1],
        "Match_Support": class_support[1],
        
        "NonMatch_Precision": class_precision[0],
        "NonMatch_Recall": class_recall[0],
        "NonMatch_F1": class_f1[0],
        "NonMatch_Support": class_support[0],
    })

chatgpt_fold_results_df = pd.DataFrame(fold_results)

chatgpt_fold_results_df.to_csv(
    f"chatgpt_{RUN_MODE}_fold_level_results_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

10. Save summary_match_nonmatch.csv

In [14]:
chatgpt_summary_df = (
    chatgpt_fold_results_df
    .groupby("Model")[
        [
            "Match_Precision",
            "Match_Recall",
            "Match_F1",
            "NonMatch_Precision",
            "NonMatch_Recall",
            "NonMatch_F1"
        ]
    ]
    .agg(["mean", "std"])
)

print(chatgpt_summary_df)

chatgpt_summary_df.to_csv(
    f"chatgpt_{RUN_MODE}_summary_match_nonmatch.csv",
    encoding="utf-8-sig"
)

                  Match_Precision           Match_Recall            Match_F1  \
                             mean       std         mean       std      mean   
Model                                                                          
ChatGPT_zero_shot        0.980559  0.026368     0.759712  0.051685  0.854982   

                            NonMatch_Precision           NonMatch_Recall  \
                        std               mean       std            mean   
Model                                                                      
ChatGPT_zero_shot  0.031904           0.954099  0.009418         0.99682   

                            NonMatch_F1            
                        std        mean       std  
Model                                              
ChatGPT_zero_shot  0.004547    0.974961  0.004886  


11. Save HAC similarity scores

In [15]:
chatgpt_hac_score_df = valid_pred_df[
    [
        "Model",
        "Fold",
        "InstanceID_1",
        "OrgName_1",
        "InstanceID_2",
        "OrgName_2",
        "TrueLabel",
        "PredictedLabel",
        "Probability_Match"
    ]
].copy()

chatgpt_hac_score_df = chatgpt_hac_score_df.rename(
    columns={"Probability_Match": "SimilarityScore"}
)

chatgpt_hac_score_df.to_csv(
    f"chatgpt_{RUN_MODE}_hac_similarity_scores.csv",
    index=False,
    encoding="utf-8-sig"
)